# CAATs de análisis de datos — 1. Calidad de datos

**Auditoría Informática — LSI — FCEQyN — UNaM**

Antes de sacar conclusiones de un conjunto de datos, el auditor tiene que poder confiar en él. En este notebook evaluamos la **calidad de los datos** extraídos del sistema de compras y pagos de una empresa ficticia, **Yerbatera del Alto Paraná S.A.**, para el ejercicio 2025.

Las pruebas se organizan según las dimensiones clásicas de calidad de datos:

| Dimensión | Pregunta | Ejemplo en esta práctica |
|---|---|---|
| Completitud | ¿Están todos los datos obligatorios? | Proveedores sin CUIT, facturas sin fecha |
| Validez | ¿Respetan formato y reglas de dominio? | Dígito verificador del CUIT, largo del CBU, fechas, importes > 0 |
| Consistencia | ¿Concuerdan entre sí? | `total = neto + IVA`, tipo de factura vs. condición de IVA |
| Unicidad | ¿Hay registros repetidos? | Mismo proveedor con dos códigos, facturas cargadas dos veces |
| Integridad referencial | ¿Las relaciones apuntan a registros existentes? | Pagos de facturas que no existen |

Cada prueba produce una tabla de **excepciones** que se guarda como papel de trabajo en `resultados/01_calidad.xlsx`.

> **Equivalencias con ACL / IDEA.** Lo que en esas herramientas son comandos, acá son operaciones de pandas: *Verify / Field Statistics* → `info()`, `describe()`, `isna()`; *Duplicates* → `duplicated()`; *Join (unmatched)* → `merge(..., indicator=True)`; *Filter / Extract* → selección booleana `df[condición]`; *Export* → `to_excel()`.

## 0. Preparación

Los archivos son extracciones (CSV) del sistema:

| Archivo | Contenido |
|---|---|
| `empleados.csv` | Legajo, datos personales, CBU de sueldo, área y usuario de sistema |
| `proveedores.csv` | Maestro de proveedores |
| `facturas.csv` | Facturas de compra recibidas |
| `pagos.csv` | Órdenes de pago emitidas |

Si la carpeta `datos/` está vacía, generarla ejecutando en una celda `!python generar_datos.py` (o, desde la terminal, `docker compose run --rm datos python generar_datos.py`).

In [1]:
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

DATOS = Path("datos")
RESULTADOS = Path("resultados")
RESULTADOS.mkdir(exist_ok=True)

hallazgos = {}  # nombre de la prueba -> DataFrame con las excepciones


def registrar(nombre, excepciones):
    """Guarda las excepciones de una prueba para el papel de trabajo y las muestra."""
    hallazgos[nombre] = excepciones
    print(f"{nombre}: {len(excepciones)} excepciones")
    return excepciones

## 1. Primera trampa: importar los datos

Leer un CSV parece trivial, pero es el primer punto donde se pierde calidad. Leamos el maestro de proveedores sin indicar tipos de dato:

In [2]:
ingenuo = pd.read_csv(DATOS / "proveedores.csv")
ingenuo[["id_proveedor", "cbu"]].head()

,id_proveedor,cbu
0,P0001,74864919970809358869
1,P0002,119862790128327645669
2,P0003,72386420400196828734
3,P0004,728278068749435143268
4,P0005,866566122274644005859


In [3]:
ingenuo.cbu.astype(str).str.len().value_counts().sort_index()

cbu
20    21
21    77
22    24
Name: count, dtype: int64

pandas interpretó el CBU como un **número**: los CBU que empiezan con `0` (código de banco `007`, `011`, ...) perdieron los ceros y parecen tener 20 o 21 dígitos. Lo mismo pasa al abrir el CSV en Excel, que además lo muestra como `2,85827E+21` y redondea los últimos dígitos.

**Regla:** los identificadores (CBU, CUIT, legajos, números de comprobante) son **texto**, aunque estén formados por dígitos. Importamos todo como texto y convertimos explícitamente solo las columnas numéricas y de fecha.

In [4]:
empleados = pd.read_csv(DATOS / "empleados.csv", dtype=str)
proveedores = pd.read_csv(DATOS / "proveedores.csv", dtype=str)
facturas = pd.read_csv(DATOS / "facturas.csv", dtype=str)
pagos = pd.read_csv(DATOS / "pagos.csv", dtype=str)

for col in ["importe_neto", "iva", "total"]:
    facturas[col] = pd.to_numeric(facturas[col])
pagos["importe"] = pd.to_numeric(pagos["importe"])

{nombre: df.shape for nombre, df in
 [("empleados", empleados), ("proveedores", proveedores), ("facturas", facturas), ("pagos", pagos)]}

{'empleados': (58, 10),
 'proveedores': (122, 8),
 'facturas': (3071, 11),
 'pagos': (2848, 7)}

## 2. Perfilado de los datos

Un perfil rápido de cada tabla: tipo, cantidad de valores nulos y de valores distintos por columna. Es el equivalente a *Verify* / *Field Statistics*.

In [5]:
def perfil(df):
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "% nulos": (df.isna().mean() * 100).round(2),
        "distintos": df.nunique(),
        "ejemplo": df.iloc[0],
    })


perfil(proveedores)

,tipo,nulos,% nulos,distintos,ejemplo
id_proveedor,str,0,0.00,122,P0001
razon_social,str,0,0.00,122,Repuestos Oberá S.A.
cuit,str,3,2.46,117,33-32081103-7
condicion_iva,str,0,0.00,2,Responsable Inscripto
cbu,str,0,0.00,120,0074864919970809358869
domicilio,str,0,0.00,120,"Av. Mitre 1707, Puerto Iguazú"
fecha_alta,str,0,0.00,119,2022-10-12
usuario_alta,str,0,0.00,5,prodriguez


In [6]:
perfil(facturas)

,tipo,nulos,% nulos,distintos,ejemplo
id_factura,str,0,0.00,3071,F00001
id_proveedor,str,0,0.00,122,P0054
tipo,str,0,0.00,2,A
punto_venta,str,0,0.00,1,2
numero,str,0,0.00,2065,1180
fecha,str,4,0.13,274,2025-12-10
importe_neto,float64,0,0.00,3060,611680.9
iva,float64,0,0.00,2629,128452.99
total,float64,0,0.00,3060,740133.89
usuario_carga,str,0,0.00,5,csuarez


In [7]:
facturas[["importe_neto", "iva", "total"]].describe().round(2)

,importe_neto,iva,total
count,3071.00,3071.00,3071.00
mean,439608.23,76041.46,515648.10
std,1477659.85,280588.47,1743031.24
min,-445602.64,-93576.55,-539179.19
25%,28981.78,2413.04,34037.20
50%,98418.23,13121.94,115074.26
75%,326988.44,51535.77,381177.23
max,37300653.23,7833137.18,45133790.41


Ya se ven señales: `cuit` y `fecha` tienen nulos, y el mínimo de los importes es **negativo**. Las pruebas siguientes lo formalizan.

## 3. Completitud

Campos obligatorios sin valor. `fecha_baja` de empleados *no* es obligatoria (vacía = empleado activo), por eso no se incluye.

In [8]:
obligatorios = {
    "proveedores": (proveedores, ["id_proveedor", "razon_social", "cuit", "cbu"]),
    "facturas": (facturas, ["id_factura", "id_proveedor", "fecha", "total"]),
    "pagos": (pagos, ["nro_orden_pago", "id_factura", "fecha_hora", "importe", "cbu_destino"]),
}
for tabla, (df, columnas) in obligatorios.items():
    for col in columnas:
        faltantes = df[df[col].isna()]
        if len(faltantes):
            registrar(f"Completitud - {tabla}.{col} vacío", faltantes)

Completitud - proveedores.cuit vacío: 3 excepciones
Completitud - facturas.fecha vacío: 4 excepciones


In [9]:
hallazgos["Completitud - proveedores.cuit vacío"]

,id_proveedor,razon_social,cuit,condicion_iva,cbu,domicilio,fecha_alta,usuario_alta
12,P0013,Logística Eldorado S.A.S.,NaN,Responsable Inscripto,2858256435456556085354,"Belgrano 927, Jardín América",2021-04-26,mruiz
49,P0050,Agroquímicos Puerto Iguazú S.H.,NaN,Responsable Inscripto,0866069795946524553139,"Rivadavia 3957, Garupá",2021-12-21,mruiz
110,P0111,Seguridad Apóstoles S.H.,NaN,Responsable Inscripto,0863019942123506926509,"Belgrano 3168, Eldorado",2018-02-26,mruiz


## 4. Validez

### 4.1 CUIT: dígito verificador

El último dígito del CUIT se calcula a partir de los 10 anteriores (módulo 11, pesos `5 4 3 2 7 6 5 4 3 2`). Un dígito verificador incorrecto indica un error de carga o un CUIT inventado.

In [10]:
def cuit_valido(cuit):
    if pd.isna(cuit):
        return None  # la ausencia ya se reportó en completitud
    digitos = cuit.replace("-", "")
    if len(digitos) != 11 or not digitos.isdigit():
        return False
    pesos = [5, 4, 3, 2, 7, 6, 5, 4, 3, 2]
    resto = 11 - sum(int(d) * p for d, p in zip(digitos[:10], pesos)) % 11
    verificador = {11: 0, 10: 9}.get(resto, resto)
    return verificador == int(digitos[10])


print(cuit_valido("20-17254359-7"), cuit_valido("20-17254359-3"))  # prueba de la función

registrar("Validez - CUIT de proveedor inválido",
          proveedores[proveedores.cuit.map(cuit_valido) == False])

True False
Validez - CUIT de proveedor inválido: 4 excepciones


,id_proveedor,razon_social,cuit,condicion_iva,cbu,domicilio,fecha_alta,usuario_alta
5,P0006,Seguridad Posadas S.R.L.,30-17333965-1,Responsable Inscripto,0140907326198269397019,"Junín 2024, Eldorado",2020-07-21,jsuarez
14,P0015,Combustibles Montecarlo S.H.,33-28459639-2,Responsable Inscripto,1911696358092932620572,"Av. Libertad 2529, Leandro N. Alem",2017-11-28,prodriguez
64,P0065,Limpieza Oberá S.A.,30-25292391-6,Responsable Inscripto,0111128407194284920419,"Belgrano 2133, Jardín América",2024-05-20,jsuarez
89,P0090,Transportes Montecarlo S.H.,20-42845252-5,Monotributo,0863821637311283414847,"Junín 1409, Eldorado",2018-01-26,jsuarez


In [ ]:
registrar("Validez - CUIT de empleado inválido",
          empleados[empleados.cuit.map(cuit_valido) == False])

### 4.2 CBU: largo y dígitos verificadores

El CBU tiene 22 dígitos en dos bloques, cada uno con su dígito verificador:
- Bloque 1 (8 dígitos): banco (3) + sucursal (4) + verificador. Pesos `7 1 3 9 7 1 3`.
- Bloque 2 (14 dígitos): cuenta (13) + verificador. Pesos `3 9 7 1 3 9 7 1 3 9 7 1 3`.

In [11]:
def verificador_bloque(digitos, pesos):
    return (10 - sum(int(d) * p for d, p in zip(digitos, pesos)) % 10) % 10


def cbu_valido(cbu):
    if pd.isna(cbu) or len(cbu) != 22 or not cbu.isdigit():
        return False
    b1, b2 = cbu[:8], cbu[8:]
    return (verificador_bloque(b1[:7], [7, 1, 3, 9, 7, 1, 3]) == int(b1[7])
            and verificador_bloque(b2[:13], [3, 9, 7, 1, 3, 9, 7, 1, 3, 9, 7, 1, 3]) == int(b2[13]))


cbu_invalidos = proveedores[~proveedores.cbu.map(cbu_valido)].assign(largo=lambda d: d.cbu.str.len())
registrar("Validez - CBU de proveedor inválido", cbu_invalidos)

Validez - CBU de proveedor inválido: 2 excepciones


,id_proveedor,razon_social,cuit,condicion_iva,cbu,domicilio,fecha_alta,usuario_alta,largo
16,P0017,Consultora Apóstoles S.R.L.,33-15955033-3,Responsable Inscripto,072772279901033436450,"Av. Mitre 2391, Garupá",2023-01-12,mgimenez,21
20,P0021,Imprenta Eldorado S.A.,30-24721618-5,Responsable Inscripto,285059614358597054169,"Rivadavia 135, Puerto Iguazú",2021-02-25,csuarez,21


### 4.3 Fechas

Convertimos la fecha con el formato esperado (`AAAA-MM-DD`) y `errors="coerce"`: lo que no se pueda interpretar queda como `NaT`. Luego clasificamos *por qué* falló.

In [ ]:
facturas["fecha_dt"] = pd.to_datetime(facturas.fecha, format="%Y-%m-%d", errors="coerce")

no_convertidas = facturas[facturas.fecha.notna() & facturas.fecha_dt.isna()].copy()
no_convertidas["motivo"] = np.where(
    no_convertidas.fecha.str.match(r"^\d{2}/\d{2}/\d{4}$"), "formato dd/mm/aaaa", "fecha inexistente")
registrar("Validez - fecha de factura con formato o valor inválido",
          no_convertidas[["id_factura", "id_proveedor", "fecha", "motivo"]])

Las fechas en formato `dd/mm/aaaa` son recuperables (es un problema de formato, no de contenido). En cambio `2026-02-30` no existe: hay que pedir el comprobante.

### 4.4 Rango: fechas fuera del período auditado

In [ ]:
fuera = facturas[(facturas.fecha_dt < "2025-01-01") | (facturas.fecha_dt > "2025-12-31")]
registrar("Validez - factura fuera del ejercicio 2025", fuera[["id_factura", "id_proveedor", "fecha", "total"]])

### 4.5 Importes no positivos

In [ ]:
registrar("Validez - factura con importe <= 0",
          facturas[facturas.total <= 0][["id_factura", "id_proveedor", "importe_neto", "iva", "total", "estado"]])

## 5. Consistencia

### 5.1 Total = neto + IVA

Comparamos con una tolerancia de un centavo para no reportar diferencias de redondeo.

In [ ]:
diferencia = (facturas.importe_neto + facturas.iva - facturas.total).round(2)
inconsistentes = facturas[diferencia.abs() > 0.01].assign(diferencia=diferencia)
registrar("Consistencia - total distinto de neto + IVA",
          inconsistentes[["id_factura", "id_proveedor", "importe_neto", "iva", "total", "diferencia"]])

### 5.2 Tipo de factura vs. condición frente al IVA

Un proveedor monotributista emite factura C (sin IVA discriminado); un responsable inscripto emite factura A a otra empresa inscripta.

In [ ]:
fp = facturas.merge(proveedores[["id_proveedor", "condicion_iva"]], on="id_proveedor", how="left")
pd.crosstab(fp.condicion_iva, fp.tipo)

In [ ]:
tipo_esperado = fp.condicion_iva.map({"Monotributo": "C", "Responsable Inscripto": "A"})
registrar("Consistencia - tipo de factura no corresponde a la condición de IVA",
          fp[fp.condicion_iva.notna() & (fp.tipo != tipo_esperado)])

Que una prueba dé **0 excepciones** también es evidencia: documenta que el control se ejecutó y fue satisfactorio.

## 6. Unicidad

### 6.1 Claves primarias

In [ ]:
for tabla, df, clave in [("empleados", empleados, "legajo"), ("proveedores", proveedores, "id_proveedor"),
                          ("facturas", facturas, "id_factura"), ("pagos", pagos, "nro_orden_pago")]:
    repetidas = df[df[clave].duplicated(keep=False)]
    if len(repetidas):
        registrar(f"Unicidad - {tabla}.{clave} repetido", repetidas)
    else:
        print(f"{tabla}.{clave}: sin repetidos")

### 6.2 Proveedores duplicados

El código de proveedor es único, pero eso no garantiza que cada proveedor esté una sola vez. Buscamos el **mismo CUIT** con códigos distintos, y comparamos razones sociales **normalizadas** (sin espacios sobrantes, mayúsculas, sin tildes ni puntuación).

In [ ]:
def normalizar(texto):
    texto = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    return " ".join(texto.upper().replace(".", " ").replace(",", " ").split())


proveedores["razon_normalizada"] = proveedores.razon_social.map(normalizar)

mismo_cuit = proveedores[proveedores.cuit.notna() & proveedores.cuit.duplicated(keep=False)]
registrar("Unicidad - proveedores con el mismo CUIT",
          mismo_cuit.sort_values("cuit")[["id_proveedor", "razon_social", "razon_normalizada", "cuit", "fecha_alta", "usuario_alta"]])

Notar que `razon_social` es distinta en cada par (mayúsculas, puntos, espacios) y por eso un control de "razón social repetida" no los habría detectado; la versión normalizada sí coincide.

### 6.3 Facturas cargadas dos veces

Una factura se identifica por proveedor + tipo + punto de venta + número, no por el código interno que asigna el sistema al cargarla.

In [ ]:
clave_comprobante = ["id_proveedor", "tipo", "punto_venta", "numero"]
duplicadas = facturas[facturas.duplicated(clave_comprobante, keep=False)]
registrar("Unicidad - factura cargada más de una vez",
          duplicadas.sort_values(clave_comprobante)[["id_factura", *clave_comprobante, "fecha", "total", "estado", "usuario_carga"]])

¡Todas figuran como **pagadas**! Este hallazgo de calidad tiene impacto económico directo; lo retomamos en el notebook 2.

## 7. Integridad referencial

Con `merge(..., how="left", indicator=True)` identificamos los registros sin contraparte (*Join unmatched* en ACL).

In [ ]:
def sin_contraparte(hijos, padres, clave):
    cruce = hijos.merge(padres[[clave]].drop_duplicates(), on=clave, how="left", indicator=True)
    return cruce[cruce._merge == "left_only"].drop(columns="_merge")


registrar("Integridad - factura de proveedor inexistente",
          sin_contraparte(facturas, proveedores, "id_proveedor")[["id_factura", "id_proveedor", "fecha", "total"]])
registrar("Integridad - pago de factura inexistente",
          sin_contraparte(pagos, facturas, "id_factura"))

## 8. Resumen y papel de trabajo

In [ ]:
resumen = pd.DataFrame(
    [(nombre.split(" - ")[0], nombre.split(" - ")[1], len(df)) for nombre, df in hallazgos.items()],
    columns=["dimensión", "prueba", "excepciones"])
resumen

In [ ]:
archivo = RESULTADOS / "01_calidad.xlsx"
with pd.ExcelWriter(archivo) as excel:
    resumen.to_excel(excel, sheet_name="Resumen", index=False)
    for i, (nombre, df) in enumerate(hallazgos.items(), start=1):
        df.to_excel(excel, sheet_name=f"{i:02d}", index=False)  # Excel limita los nombres de hoja a 31 caracteres
print(f"Papel de trabajo: {archivo}")

## 9. Ejercicios

1. **Conclusión de auditoría.** Con el resumen anterior, ¿los datos son confiables para hacer pruebas sobre los pagos? ¿Qué registros excluirían o corregirían antes de seguir, y qué le pedirían a la empresa?
2. **Fechas recuperables.** Convertir las fechas en formato `dd/mm/aaaa` (pista: `pd.to_datetime(..., format="%d/%m/%Y")`) y volver a calcular cuántas facturas quedan sin fecha válida.
3. **Validez en empleados.** Aplicar `cbu_valido` a los CBU de `empleados`. ¿Qué resultado obtienen?
4. **Pagos.** Construir `fecha_dt` para `pagos.fecha_hora` y verificar que todas las fechas sean válidas.
5. **Completitud cruzada.** ¿Hay usuarios en `facturas.usuario_carga` o `pagos.usuario_aprueba` que no existan en `empleados.usuario`?
6. **Otro conjunto de datos.** Regenerar los datos con otra semilla (en una celda: `!python generar_datos.py --semilla 7`) y volver a ejecutar todo el notebook (*Run All*). ¿Las pruebas siguen funcionando sin cambios? Esa es la ventaja de una CAAT frente a una revisión manual.